In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(f"Raw data shape: {df.shape}")
df.head(3)

Raw data shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes


In [2]:
# Convert to numeric — blanks become NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Fill NaN with MonthlyCharges (these are new customers, tenure=0)
df['TotalCharges'] = df['TotalCharges'].fillna(df['MonthlyCharges'])

print(f"TotalCharges nulls remaining: {df['TotalCharges'].isnull().sum()}")
print(f"TotalCharges dtype: {df['TotalCharges'].dtype}")

TotalCharges nulls remaining: 0
TotalCharges dtype: float64


In [3]:
df['Churn'] = (df['Churn'] == 'Yes').astype(int)
print(f"Churn value counts:\n{df['Churn'].value_counts()}")

Churn value counts:
Churn
0    5174
1    1869
Name: count, dtype: int64


In [4]:
# Yes/No columns → 1/0
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling',
               'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
               'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']

for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0, 'No phone service': 0, 'No internet service': 0})

# Gender → binary
df['gender'] = (df['gender'] == 'Male').astype(int)

print("Binary encoding done.")
print(df[binary_cols].head(3))

Binary encoding done.
   Partner  Dependents  PhoneService  PaperlessBilling  MultipleLines  \
0        1           0             0                 1              0   
1        0           0             1                 0              0   
2        0           0             1                 1              0   

   OnlineSecurity  OnlineBackup  DeviceProtection  TechSupport  StreamingTV  \
0               0             1                 0            0            0   
1               1             0                 1            0            0   
2               1             1                 0            0            0   

   StreamingMovies  
0                0  
1                0  
2                0  


In [5]:
# Contract, InternetService, PaymentMethod
df = pd.get_dummies(df, columns=['Contract', 'InternetService', 'PaymentMethod'], drop_first=False)

# Convert any bool columns to int
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

print(f"Shape after one-hot encoding: {df.shape}")
print([c for c in df.columns if any(x in c for x in ['Contract', 'Internet', 'Payment'])])

Shape after one-hot encoding: (7043, 28)
['Contract_Month-to-month', 'Contract_One year', 'Contract_Two year', 'InternetService_DSL', 'InternetService_Fiber optic', 'InternetService_No', 'PaymentMethod_Bank transfer (automatic)', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']


In [6]:
# 1. Tenure buckets
df['tenure_group'] = pd.cut(df['tenure'],
                             bins=[0, 12, 24, 48, 72],
                             labels=['New', 'Developing', 'Mature', 'Loyal'],
                             include_lowest=True)
df['tenure_group'] = df['tenure_group'].astype(str)

# 2. Avg monthly spend proxy (TotalCharges / tenure)
df['avg_monthly_spend'] = np.where(
    df['tenure'] > 0,
    df['TotalCharges'] / df['tenure'],
    df['MonthlyCharges']
)

# 3. Charge per service (how much they pay relative to services used)
service_cols = ['PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
                'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
df['num_services'] = df[service_cols].sum(axis=1)
df['charge_per_service'] = np.where(
    df['num_services'] > 0,
    df['MonthlyCharges'] / df['num_services'],
    df['MonthlyCharges']
)

# 4. High bill flag — paying more than average
df['high_monthly_charges'] = (df['MonthlyCharges'] > df['MonthlyCharges'].median()).astype(int)

# 5. Customer LTV (lifetime value proxy)
df['LTV'] = df['tenure'] * df['MonthlyCharges']

# 6. Engagement score (more services = more engaged)
df['engagement_score'] = df[service_cols].sum(axis=1)

print("New features created:")
new_features = ['tenure_group', 'avg_monthly_spend', 'num_services',
                'charge_per_service', 'high_monthly_charges', 'LTV', 'engagement_score']
print(df[new_features].describe())

New features created:
       avg_monthly_spend  num_services  charge_per_service  \
count        7043.000000   7043.000000         7043.000000   
mean           64.762906      3.362914           22.509972   
std            30.189796      2.062031           11.576218   
min            13.775000      0.000000           10.416667   
25%            35.935156      1.000000           15.700000   
50%            70.337500      3.000000           19.750000   
75%            90.174158      5.000000           24.525000   
max           121.400000      8.000000           72.250000   

       high_monthly_charges          LTV  engagement_score  
count           7043.000000  7043.000000       7043.000000  
mean               0.499077  2279.581350          3.362914  
std                0.500035  2264.729447          2.062031  
min                0.000000     0.000000          0.000000  
25%                0.000000   394.000000          1.000000  
50%                0.000000  1393.600000          3.0

In [7]:
# LTV segments (based on median split)
ltv_median = df['LTV'].median()
df['value_segment'] = np.where(df['LTV'] >= ltv_median, 'High Value', 'Low Value')

# Risk segments — placeholder, will be updated after model training
# For now based on EDA heuristics
def heuristic_risk(row):
    score = 0
    if row.get('Contract_Month-to-month', 0) == 1:
        score += 3
    if row['tenure'] < 12:
        score += 2
    if row['high_monthly_charges'] == 1:
        score += 1
    if row.get('InternetService_Fiber optic', 0) == 1:
        score += 1
    if score >= 5:
        return 'High Risk'
    elif score >= 3:
        return 'Medium Risk'
    else:
        return 'Low Risk'

df['heuristic_risk'] = df.apply(heuristic_risk, axis=1)
print(df['heuristic_risk'].value_counts())
print(df['value_segment'].value_counts())

heuristic_risk
Low Risk       3157
High Risk      3128
Medium Risk     758
Name: count, dtype: int64
value_segment
High Value    3522
Low Value     3521
Name: count, dtype: int64


In [8]:
# Drop customerID (not a feature) and tenure_group string column
# Keep customerID separately for reference
customer_ids = df['customerID'].copy()

# Columns to drop from model features
drop_cols = ['customerID', 'tenure_group', 'heuristic_risk', 'value_segment', 'LTV']

df_model = df.drop(columns=drop_cols)

# Verify all columns are numeric
non_numeric = df_model.select_dtypes(exclude=[np.number]).columns.tolist()
print(f"Non-numeric columns remaining: {non_numeric}")

# Save processed dataset
df_model.to_csv('data/processed/telco_churn_processed.csv', index=False)

# Save full dataset with segments for Streamlit
df['customerID'] = customer_ids
df.to_csv('data/processed/telco_churn_full.csv', index=False)

print(f"\nProcessed dataset shape: {df_model.shape}")
print(f"Full dataset shape: {df.shape}")
print("\nFeature columns for model:")
print(list(df_model.columns))

Non-numeric columns remaining: []

Processed dataset shape: (7043, 32)
Full dataset shape: (7043, 37)

Feature columns for model:
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling', 'MonthlyCharges', 'TotalCharges', 'Churn', 'Contract_Month-to-month', 'Contract_One year', 'Contract_Two year', 'InternetService_DSL', 'InternetService_Fiber optic', 'InternetService_No', 'PaymentMethod_Bank transfer (automatic)', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check', 'avg_monthly_spend', 'num_services', 'charge_per_service', 'high_monthly_charges', 'engagement_score']
